BASELINE VALIDATION CODE

Llama-3.2-1B-Instruct

In [ ]:
import pandas as pd
import torch
import gc
from datasets import load_dataset
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from google.colab import drive
drive.mount('/content/drive')

#Download Dataset
print("Loading MILU Dataset...")
dataset = load_dataset("ai4bharat/MILU", data_dir="Choose_Your_Language", split="Choose_Test_Or_Validation_Split", token=True)
df = dataset.to_pandas()

target_map = {
    "option1": "A",
    "option2": "B",
    "option3": "C",
    "option4": "D"
}

if 'target' in df.columns:
    df['target_clean'] = df['target'].str.strip().str.lower().map(target_map)
else:
    raise KeyError("The 'target' column was not found in the MILU dataset.")

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

token_ids = {
    "A": tokenizer.encode("A", add_special_tokens=False)[0],
    "B": tokenizer.encode("B", add_special_tokens=False)[0],
    "C": tokenizer.encode("C", add_special_tokens=False)[0],
    "D": tokenizer.encode("D", add_special_tokens=False)[0]
}

print(f"Loading {MODEL_NAME} via vLLM...")
llm = LLM(
    model=MODEL_NAME,
    dtype="half",
    gpu_memory_utilization=0.70,
    max_model_len=4096,
    enforce_eager=True,
    disable_custom_all_reduce=True
)

prompts = []
for _, row in df.iterrows():
    content = f"Question: {row['question']}\nOptions:\nA) {row['option1']}\nB) {row['option2']}\nC) {row['option3']}\nD) {row['option4']}"
    messages = [
        {"role": "system", "content": "You are an expert test-taking AI. Respond ONLY with the single letter of the correct option (A, B, C, or D)."},
        {"role": "user", "content": content}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompts.append(text)

sampling_params = SamplingParams(temperature=0.0, max_tokens=1, logprobs=20)

print("Scoring zero-shot log-likelihoods for Baseline...")
outputs = llm.generate(prompts, sampling_params, use_tqdm=True)

llama_preds = []
for output in outputs:
    if output.outputs[0].logprobs:
        first_token_logprobs = output.outputs[0].logprobs[0]

        scores = {}
        for letter, t_id in token_ids.items():
            if t_id in first_token_logprobs:
                scores[letter] = first_token_logprobs[t_id].logprob
            else:
                scores[letter] = -999.0

        best_letter = max(scores, key=scores.get)

        if all(v == -999.0 for v in scores.values()):
            gen_text = output.outputs[0].text.strip().upper()
            best_letter = gen_text if gen_text in ["A", "B", "C", "D"] else "A"
    else:
        gen_text = output.outputs[0].text.strip().upper()
        best_letter = gen_text if gen_text in ["A", "B", "C", "D"] else "A"

    llama_preds.append(best_letter)

df['llama_baseline_pred'] = llama_preds

valid_df = df[df['target_clean'].isin(["A", "B", "C", "D"])]
correct = (valid_df['llama_baseline_pred'] == valid_df['target_clean']).sum()
total = len(valid_df)
accuracy = (correct / total) * 100 if total > 0 else 0

print("\n=== LLAMA-3.2-1B ZERO-SHOT BASELINE METRICS ===")
print(f"Total Evaluated: {total}")
print(f"Baseline Accuracy: {accuracy:.2f}%")

output_path = "PATH_WHERE_BASELINE_RESULT_IS_STORED"
df.to_csv(output_path, index=False)
print(f"Final results appended and saved to {output_path}")

del llm
gc.collect()
torch.cuda.empty_cache()

Llama-3.2-3B-Instruct

In [ ]:
import pandas as pd
import torch
import gc
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import drive
drive.mount('/content/drive')

#Download Dataset
print("Loading MILU Dataset...")
dataset = load_dataset("ai4bharat/MILU", data_dir="Choose_Your_Language", split="Choose_Test_Or_Validation_Split", token=True)
df = dataset.to_pandas()

target_map = {"option1": "A", "option2": "B", "option3": "C", "option4": "D"}
df['target_clean'] = df['target'].str.strip().str.lower().map(target_map)

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
print(f"Loading {MODEL_NAME} via Hugging Face...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="cuda", trust_remote_code=True)

token_ids = {
    "A": tokenizer.encode("A", add_special_tokens=False)[-1],
    "B": tokenizer.encode("B", add_special_tokens=False)[-1],
    "C": tokenizer.encode("C", add_special_tokens=False)[-1],
    "D": tokenizer.encode("D", add_special_tokens=False)[-1]
}

llama_preds = []
print("Scoring zero-shot log-likelihoods for Baseline...")
for _, row in df.iterrows():
    content = f"Question: {row['question']}\nOptions:\nA) {row['option1']}\nB) {row['option2']}\nC) {row['option3']}\nD) {row['option4']}"
    messages = [
        {"role": "system", "content": "You are an expert test-taking AI. Respond ONLY with the single letter of the correct option (A, B, C, or D)."},
        {"role": "user", "content": content}
    ]
    try:
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        # Fallback if model lacks a chat template
        prompt = messages[0]["content"] + "\n\n" + messages[1]["content"] + "\nAnswer: "

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        logits = model(**inputs).logits[0, -1, :]
        scores = {letter: logits[t_id].item() for letter, t_id in token_ids.items()}
        best_letter = max(scores, key=scores.get)

    llama_preds.append(best_letter)

df['llama_baseline_pred'] = llama_preds

valid_df = df[df['target_clean'].isin(["A", "B", "C", "D"])]
correct = (valid_df['llama_baseline_pred'] == valid_df['target_clean']).sum()
total = len(valid_df)
accuracy = (correct / total) * 100 if total > 0 else 0

print("\n=== BASELINE METRICS ===")
print(f"Total Evaluated: {total}")
print(f"Baseline Accuracy: {accuracy:.2f}%")

output_path = f"OUTPUT_PATH_HERE_{MODEL_NAME.split('/')[-1]}_baseline.csv"
df.to_csv(output_path, index=False)
print(f"Final results saved to {output_path}")

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

Llama-3.1-8B-Instruct

In [ ]:
import pandas as pd
import torch
import gc
from datasets import load_dataset
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from google.colab import drive
drive.mount('/content/drive')

#Download Dataset
print("Loading MILU Dataset...")
dataset = load_dataset("ai4bharat/MILU", data_dir="Choose_Your_Language", split="Choose_Test_Or_Validation_Split", token=True)
df = dataset.to_pandas()

target_map = {"option1": "A", "option2": "B", "option3": "C", "option4": "D"}
df['target_clean'] = df['target'].str.strip().str.lower().map(target_map)

MODEL_NAME = "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4"
print(f"Loading {MODEL_NAME} via vLLM...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

token_ids = {
    "A": tokenizer.encode("A", add_special_tokens=False)[-1],
    "B": tokenizer.encode("B", add_special_tokens=False)[-1],
    "C": tokenizer.encode("C", add_special_tokens=False)[-1],
    "D": tokenizer.encode("D", add_special_tokens=False)[-1]
}

llm = LLM(
    model=MODEL_NAME,
    quantization="awq",
    dtype="half",
    gpu_memory_utilization=0.90,
    max_model_len=4096,
    enforce_eager=True,
    disable_custom_all_reduce=True
)

prompts = []
for _, row in df.iterrows():
    content = f"Question: {row['question']}\nOptions:\nA) {row['option1']}\nB) {row['option2']}\nC) {row['option3']}\nD) {row['option4']}"
    messages = [
        {"role": "system", "content": "You are an expert test-taking AI. Respond ONLY with the single letter of the correct option (A, B, C, or D)."},
        {"role": "user", "content": content}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompts.append(text)

sampling_params = SamplingParams(temperature=0.0, max_tokens=1, logprobs=5)

print("Batch generating zero-shot log-likelihoods...")
outputs = llm.generate(prompts, sampling_params, use_tqdm=True)

llama_preds = []
for output in outputs:
    scores = {letter: -999.0 for letter in ["A", "B", "C", "D"]}

    if output.outputs[0].logprobs:
        first_token_logprobs = output.outputs[0].logprobs[0]
        for letter, t_id in token_ids.items():
            if t_id in first_token_logprobs:
                scores[letter] = first_token_logprobs[t_id].logprob

    best_letter = max(scores, key=scores.get)

    # Fallback if the model outputs a completely unexpected token
    if all(v == -999.0 for v in scores.values()):
        gen_text = output.outputs[0].text.strip().upper()
        best_letter = gen_text if gen_text in ["A", "B", "C", "D"] else "A"

    llama_preds.append(best_letter)

df['llama_baseline_pred'] = llama_preds

valid_df = df[df['target_clean'].isin(["A", "B", "C", "D"])]
correct = (valid_df['llama_baseline_pred'] == valid_df['target_clean']).sum()
total = len(valid_df)
accuracy = (correct / total) * 100 if total > 0 else 0

print("\n=== vLLM ZERO-SHOT BASELINE METRICS ===")
print(f"Total Evaluated: {total}")
print(f"Baseline Accuracy: {accuracy:.2f}%")

output_path = f"OUTPUT_PATH_HERE_{MODEL_NAME.split('/')[-1]}_baseline.csv"
df.to_csv(output_path, index=False)
print(f"Final results saved to {output_path}")

del llm
gc.collect()
torch.cuda.empty_cache()

Aya-23-8B

In [ ]:
import pandas as pd
import torch
import gc
from datasets import load_dataset
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from google.colab import drive
drive.mount('/content/drive')

#Download Dataset
print("Loading MILU Dataset...")
dataset = load_dataset("ai4bharat/MILU", data_dir="Choose_Your_Language", split="Choose_Test_Or_Validation_Split", token=True)
df = dataset.to_pandas()

target_map = {"option1": "A", "option2": "B", "option3": "C", "option4": "D"}
df['target_clean'] = df['target'].str.strip().str.lower().map(target_map)

MODEL_NAME = "alijawad07/aya-23-8B-AWQ-GEMM"
print(f"Loading {MODEL_NAME} via vLLM...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

token_ids = {
    "A": tokenizer.encode("A", add_special_tokens=False)[-1],
    "B": tokenizer.encode("B", add_special_tokens=False)[-1],
    "C": tokenizer.encode("C", add_special_tokens=False)[-1],
    "D": tokenizer.encode("D", add_special_tokens=False)[-1]
}

llm = LLM(
    model=MODEL_NAME,
    quantization="awq",
    dtype="half",
    gpu_memory_utilization=0.90,
    max_model_len=4096,
    enforce_eager=True,
    disable_custom_all_reduce=True
)

prompts = []
for _, row in df.iterrows():
    content = f"Question: {row['question']}\nOptions:\nA) {row['option1']}\nB) {row['option2']}\nC) {row['option3']}\nD) {row['option4']}"
    messages = [
        {"role": "system", "content": "You are an expert test-taking AI. Respond ONLY with the single letter of the correct option (A, B, C, or D)."},
        {"role": "user", "content": content}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompts.append(text)

sampling_params = SamplingParams(temperature=0.0, max_tokens=1, logprobs=5)

print("Batch generating zero-shot log-likelihoods...")
outputs = llm.generate(prompts, sampling_params, use_tqdm=True)

llama_preds = []
for output in outputs:
    scores = {letter: -999.0 for letter in ["A", "B", "C", "D"]}

    if output.outputs[0].logprobs:
        first_token_logprobs = output.outputs[0].logprobs[0]
        for letter, t_id in token_ids.items():
            if t_id in first_token_logprobs:
                scores[letter] = first_token_logprobs[t_id].logprob

    best_letter = max(scores, key=scores.get)

    # Fallback if the model outputs a completely unexpected token
    if all(v == -999.0 for v in scores.values()):
        gen_text = output.outputs[0].text.strip().upper()
        best_letter = gen_text if gen_text in ["A", "B", "C", "D"] else "A"

    llama_preds.append(best_letter)

df['llama_baseline_pred'] = llama_preds

valid_df = df[df['target_clean'].isin(["A", "B", "C", "D"])]
correct = (valid_df['llama_baseline_pred'] == valid_df['target_clean']).sum()
total = len(valid_df)
accuracy = (correct / total) * 100 if total > 0 else 0

print("\n=== vLLM ZERO-SHOT BASELINE METRICS ===")
print(f"Total Evaluated: {total}")
print(f"Baseline Accuracy: {accuracy:.2f}%")

output_path = f"OUTPUT_PATH_HERE_{MODEL_NAME.split('/')[-1]}_baseline.csv"
df.to_csv(output_path, index=False)
print(f"Final results saved to {output_path}")

del llm
gc.collect()
torch.cuda.empty_cache()

Gemma-2-2B-it

In [ ]:
import pandas as pd
import torch
import gc
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import drive
drive.mount('/content/drive')

#Download Dataset
print("Loading MILU Dataset...")
dataset = load_dataset("ai4bharat/MILU", data_dir="Choose_Your_Language", split="Choose_Test_Or_Validation_Split", token=True)
df = dataset.to_pandas()

target_map = {"option1": "A", "option2": "B", "option3": "C", "option4": "D"}
df['target_clean'] = df['target'].str.strip().str.lower().map(target_map)

MODEL_NAME = "google/gemma-2-2b-it"
print(f"Loading {MODEL_NAME} via Hugging Face...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="cuda", trust_remote_code=True)

token_ids = {
    "A": tokenizer.encode("A", add_special_tokens=False)[-1],
    "B": tokenizer.encode("B", add_special_tokens=False)[-1],
    "C": tokenizer.encode("C", add_special_tokens=False)[-1],
    "D": tokenizer.encode("D", add_special_tokens=False)[-1]
}

llama_preds = []
print("Scoring zero-shot log-likelihoods for Baseline...")
for _, row in df.iterrows():
    content = f"Question: {row['question']}\nOptions:\nA) {row['option1']}\nB) {row['option2']}\nC) {row['option3']}\nD) {row['option4']}"
    messages = [
        {"role": "system", "content": "You are an expert test-taking AI. Respond ONLY with the single letter of the correct option (A, B, C, or D)."},
        {"role": "user", "content": content}
    ]
    try:
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        # Fallback if model lacks a chat template
        prompt = messages[0]["content"] + "\n\n" + messages[1]["content"] + "\nAnswer: "

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        logits = model(**inputs).logits[0, -1, :]
        scores = {letter: logits[t_id].item() for letter, t_id in token_ids.items()}
        best_letter = max(scores, key=scores.get)

    llama_preds.append(best_letter)

df['llama_baseline_pred'] = llama_preds

valid_df = df[df['target_clean'].isin(["A", "B", "C", "D"])]
correct = (valid_df['llama_baseline_pred'] == valid_df['target_clean']).sum()
total = len(valid_df)
accuracy = (correct / total) * 100 if total > 0 else 0

print("\n=== BASELINE METRICS ===")
print(f"Total Evaluated: {total}")
print(f"Baseline Accuracy: {accuracy:.2f}%")

output_path = f"OUTPUT_PATH_HERE_{MODEL_NAME.split('/')[-1]}_baseline.csv"
df.to_csv(output_path, index=False)
print(f"Final results saved to {output_path}")

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

Sarvam-1

In [ ]:
import pandas as pd
import torch
import gc
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import drive
drive.mount('/content/drive')

#Download Dataset
print("Loading MILU Dataset...")
dataset = load_dataset("ai4bharat/MILU", data_dir="Choose_Your_Language", split="Choose_Test_Or_Validation_Split", token=True)
df = dataset.to_pandas()

target_map = {"option1": "A", "option2": "B", "option3": "C", "option4": "D"}
df['target_clean'] = df['target'].str.strip().str.lower().map(target_map)

MODEL_NAME = "sarvamai/sarvam-1"
print(f"Loading {MODEL_NAME} via Hugging Face...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="cuda", trust_remote_code=True)

token_ids = {
    "A": tokenizer.encode("A", add_special_tokens=False)[-1],
    "B": tokenizer.encode("B", add_special_tokens=False)[-1],
    "C": tokenizer.encode("C", add_special_tokens=False)[-1],
    "D": tokenizer.encode("D", add_special_tokens=False)[-1]
}

llama_preds = []
print("Scoring zero-shot log-likelihoods for Baseline...")
for _, row in df.iterrows():
    content = f"Question: {row['question']}\nOptions:\nA) {row['option1']}\nB) {row['option2']}\nC) {row['option3']}\nD) {row['option4']}"
    messages = [
        {"role": "system", "content": "You are an expert test-taking AI. Respond ONLY with the single letter of the correct option (A, B, C, or D)."},
        {"role": "user", "content": content}
    ]
    try:
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        # Fallback if model lacks a chat template
        prompt = messages[0]["content"] + "\n\n" + messages[1]["content"] + "\nAnswer: "

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        logits = model(**inputs).logits[0, -1, :]
        scores = {letter: logits[t_id].item() for letter, t_id in token_ids.items()}
        best_letter = max(scores, key=scores.get)

    llama_preds.append(best_letter)

df['llama_baseline_pred'] = llama_preds

valid_df = df[df['target_clean'].isin(["A", "B", "C", "D"])]
correct = (valid_df['llama_baseline_pred'] == valid_df['target_clean']).sum()
total = len(valid_df)
accuracy = (correct / total) * 100 if total > 0 else 0

print("\n=== BASELINE METRICS ===")
print(f"Total Evaluated: {total}")
print(f"Baseline Accuracy: {accuracy:.2f}%")

output_path = f"OUTPUT_PATH_HERE_{MODEL_NAME.split('/')[-1]}_baseline.csv"
df.to_csv(output_path, index=False)
print(f"Final results saved to {output_path}")

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()